# Redline-LLM: Fine-Tuning AI Governance & Crisis Negotiator

This notebook trains a domain-specific Large Language Model (**Redline-LLM**) on the **Redline Protocol** crisis simulation and policy negotiation datasets using **QLoRA (4-bit quantization)** on a single free GPU (e.g. Google Colab T4 or A100).

## 1. Check GPU & Environment

In [ ]:
!nvidia-smi

## 2. Install Required Libraries

In [ ]:
!pip install -q torch transformers datasets peft trl accelerate bitsandbytes scipy protobuf sentencepiece

## 3. Clone Repository or Upload Training Datasets
Upload the `training/data/` directory or clone your repository.

In [ ]:
# Verify files exist
!ls -la training/data/

## 4. Run QLoRA Supervised Fine-Tuning (SFT)
Target Model: `Qwen/Qwen2.5-7B-Instruct` or `meta-llama/Meta-Llama-3.1-8B-Instruct`

In [ ]:
!python training/train_lora.py \
    --base_model "Qwen/Qwen2.5-7B-Instruct" \
    --train_data "training/data/redline_sft_train.jsonl" \
    --val_data "training/data/redline_sft_val.jsonl" \
    --output_dir "models/redline-llm-lora" \
    --epochs 3 \
    --batch_size 2 \
    --grad_accum 4 \
    --lr 2e-4

## 5. Direct Preference Optimization (DPO)
Align the model's policy to prefer multilateral consensus and rapid containment over unilateral escalation.

In [ ]:
!python training/train_dpo.py \
    --base_model "Qwen/Qwen2.5-7B-Instruct" \
    --adapter_path "models/redline-llm-lora" \
    --dpo_data "training/data/redline_dpo_train.jsonl" \
    --output_dir "models/redline-llm-dpo" \
    --epochs 2

## 6. Merge LoRA Weights into Standalone SafeTensors Model

In [ ]:
!python training/export_model.py \
    --base_model "Qwen/Qwen2.5-7B-Instruct" \
    --adapter_path "models/redline-llm-dpo" \
    --output_dir "models/redline-llm-final"

## 7. Interactive Test Generation

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "models/redline-llm-final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")

messages = [
    {"role": "system", "content": "You are the Chief AI Policy Advisor for the Federal Republic of Alerion."},
    {"role": "user", "content": "An autonomous grid desynchronization crisis has spread to 3 neighboring states. What action do we take?"}
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=400, temperature=0.3)

print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))
